# 💧 水力発電候補地選定システム V2

---

## 🎯 このシステムでできること

**地名を入力するだけで、その地域に水力発電所を建設できそうな場所を自動で探し出します。**

```
[入力] 地名を入力 → [分析] 地形・河川・インフラを分析 → [出力] 最適な発電所候補地を提案
```

### 出力されるもの
- 📍 **地図**: 候補地の位置を色分けして表示
- 📊 **グラフ**: 発電量・落差の比較
- 📄 **データ**: CSV形式で詳細データを保存

---

## 🆕 V2で改良されたポイント

| 項目 | 旧バージョン | V2（改良版） |
|------|------------|-------------|
| **流量推定** | 河川タイプから固定値 | 集水域面積×降水量から計算 |
| **インフラ考慮** | なし | 道路・送電線への距離を評価 |
| **環境保護** | なし | 国立公園・保護区を自動で除外 |
| **既存ダム** | なし | 水利権競合リスクを評価 |
| **発電量精度** | 過小評価の傾向 | 現実的な小〜中規模水力相当 |

---

## 📚 水力発電の基礎知識

### 水力発電の仕組み

水力発電は、高い場所から低い場所へ水を落とし、その力でタービンを回して電気を作ります。

```
     ⛰️ 水源（高い場所）
        ↓
     💧 取水口（水を集める）
        ↓ ← 水路・導水管
     ⚡ 発電所（低い場所）
```

### 発電量を決める3つの要素

$$発電量(kW) = 9.8 \times 流量(m³/s) \times 落差(m) \times 効率$$

| 要素 | 説明 | 大きいほど |
|-----|------|----------|
| **流量** | 1秒あたりに流れる水の量 | 発電量が増える |
| **落差** | 水源と発電所の高低差 | 発電量が増える |
| **効率** | 水車・発電機の性能（通常80%程度） | 発電量が増える |

### 水力発電所の規模

| 分類 | 発電量 | 例 |
|-----|--------|----|
| マイクロ水力 | 〜100 kW | 農業用水路の落差を利用 |
| 小水力 | 100 kW〜1 MW | 山間部の小河川 |
| 中規模水力 | 1 MW〜10 MW | 地方の中小河川 |
| 大規模水力 | 10 MW〜 | 大型ダム |

**このシステムは主に小水力〜中規模水力の候補地を探索します。**

---

## 🔧 システムの技術的な仕組み

### STEP 1: 探索範囲の決定

入力された地名から行政区画の境界を取得し、その範囲内のみを分析対象とします。

- **使用API**: Nominatim API（OpenStreetMap）
- **取得データ**: 境界ポリゴン（数千〜数万点の座標）

### STEP 2: 地形データの取得

境界内にグリッド（格子状の点）を配置し、各点の標高を取得します。

- **使用API**: Open-Elevation API
- **取得データ**: 各グリッド点の標高（メートル）

### STEP 3: 河川データの取得と流量推定

OpenStreetMapから河川データを取得し、各河川の流量を推定します。

**流量の推定式（V2の改良点）**:
$$流量(m³/s) = \frac{集水域面積(km²) \times 年間降水量(mm) \times 流出係数}{365 \times 24 \times 3600}$$

| パラメータ | 説明 |
|-----------|------|
| 集水域面積 | 河川の長さから理論的に推定 |
| 年間降水量 | 都道府県別のデータを使用（例：長野県は1000mm） |
| 流出係数 | 山岳地域=0.7、丘陵=0.5、平野=0.3 |

### STEP 4: 追加データの取得（V2新機能）

| データ | 用途 |
|--------|------|
| 🛤️ 道路 | 建設資材の輸送、保守のアクセス性 |
| ⚡ 送電線 | 発電した電気を送る系統への接続性 |
| 🏞️ 保護区域 | 国立公園等は候補から除外 |
| 🏗️ 既存ダム | 水利権の競合リスクを評価 |

### STEP 5: 候補地のスコアリング

各グリッド点を評価し、スコアの高い場所を候補地として選定します。

**水源候補のスコア**:
```
スコア = 標高(30%) + 河川近接性(25%) + 流量(20%) + インフラ(15%) + 地質安定性(10%)
```

**発電所候補のスコア**:
```
スコア = 低標高(30%) + 平坦度(25%) + インフラ(25%) + 河川近接性(20%)
```

### STEP 6: 最適な組み合わせの探索

水源・取水口・発電所の全組み合わせを評価し、発電量の大きい順にランク付けします。

**物理的制約のチェック**:
- 標高の順序: 水源 > 取水口 > 発電所
- 有効落差: 10m以上
- 総水路長: 現実的な範囲

---

## 🚀 使い方

### ステップ1: 対象地域を設定

下のセルの `LOCATION_NAME` を変更してください。

```python
LOCATION_NAME = "松本市"  # ← ここを変更
```

### ステップ2: 全セルを実行

`Shift + Enter` で順番に実行するか、メニューから「すべてのセルを実行」を選択。

### ステップ3: 結果を確認

- 地図がNotebook内に表示されます
- 詳細データは `deta/yyyymmddhhmm/` フォルダに保存されます

---

In [ ]:
# ============================================================
# 🔧 設定セル
# ============================================================
# 対象地域を設定してください
LOCATION_NAME = "松本市"

# 詳細設定（通常は変更不要）
GRID_SIZE = 20           # グリッドサイズ（大きいほど詳細、計算時間増）
CANDIDATES_PER_TYPE = 20 # 各タイプの候補数
TOP_N = 10               # 出力する上位組合せ数

print(f"対象地域: {LOCATION_NAME}")
print(f"グリッドサイズ: {GRID_SIZE}x{GRID_SIZE}")
print(f"出力候補数: 上位{TOP_N}組")

In [ ]:
# ============================================================
# 🔍 分析実行セル
# ============================================================
from hydro_selector_v2 import HydroSiteSelectorV2

# インスタンス作成
selector = HydroSiteSelectorV2(LOCATION_NAME)

# 分析実行
map_result, fig_result = selector.run_analysis(
    grid_size=GRID_SIZE,
    candidates_per_type=CANDIDATES_PER_TYPE,
    top_n=TOP_N
)

# 結果保存
if selector.best_combinations:
    output_dir, saved_files = selector.save_results(map_result, fig_result)
    print(f"\n✓ 保存先: {output_dir}")
    print(f"✓ 保存ファイル数: {len(saved_files)}")

In [ ]:
# ============================================================
# 📊 結果表示セル
# ============================================================
if selector.best_combinations:
    print("=" * 60)
    print(f"上位{len(selector.best_combinations)}組の候補")
    print("=" * 60)
    print(f"{'順位':<4} {'発電量':>10} {'落差':>8} {'流量':>10}")
    print("-" * 60)
    for i, c in enumerate(selector.best_combinations, 1):
        print(f"#{i:<3} {c['power_kw']:>8.0f} kW {c['head']:>6.0f} m {c['flow']:>8.2f} m³/s")
else:
    print("候補が見つかりませんでした。")

In [ ]:
# ============================================================
# 🗺️ 地図表示セル
# ============================================================
# 候補地を地図上に表示します
# - 赤: 第1位の組合せ
# - 青: 第2位の組合せ
# - 緑: 第3位の組合せ
map_result

---

## 📁 出力ファイルの説明

結果は `deta/yyyymmddhhmm/` フォルダに保存されます：

| ファイル | 内容 | 開き方 |
|----------|------|--------|
| `hydro_map_*.html` | 候補地マップ | ブラウザで開く |
| `hydro_sites_*.csv` | 候補地の詳細データ | Excelで開く |
| `summary_*.txt` | 分析結果のサマリー | メモ帳で開く |
| `1_Elevation_*.png` | 標高分布図 | 画像ビューアで開く |
| `2_Power_*.png` | 発電量比較グラフ | 画像ビューアで開く |
| `3_Head_*.png` | 落差比較グラフ | 画像ビューアで開く |
| `4_Profile_*.png` | 施設配置図 | 画像ビューアで開く |

---

## ⚠️ 注意事項

- このシステムの結果は**参考値**です
- 実際の建設には詳細な現地調査、許認可申請が必要です
- 河川流量は推定値であり、実測値とは異なる場合があります
- 計算時間は地域の広さによって変わります（通常5〜15分程度）

---

## 📞 技術情報

### 使用API
- **Nominatim API**: 地名から座標・境界を取得
- **Open-Elevation API**: 標高データの取得
- **Overpass API**: 河川・道路・送電線・保護区域データの取得

### 使用ライブラリ
- folium（地図可視化）
- matplotlib（グラフ）
- pandas, numpy（データ処理）
- shapely（地理計算）
- geopy（距離計算）